In [51]:
# Importations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb


In [52]:
# Chargement des données
df_full = pd.read_csv("donnees_finales.csv", low_memory=False)
print(f"Chargement de {len(df_full):,} lignes du CSV.")

target = 'conso_5_usages_ef'

df = df_full[df_full[target].notna()].copy()

n_sample = min(700_000, len(df))
df = df.sample(n=n_sample, random_state=42)
print(f"Échantillon de {n_sample:,} lignes prélevé.")

for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


Chargement de 1,702,447 lignes du CSV.
Échantillon de 700,000 lignes prélevé.
0: conso_5_usages_ef
1: version_dpe
2: qualite_isolation_murs
3: type_batiment
4: classe_altitude
5: type_installation_ecs
6: type_energie_principale_chauffage
7: qualite_isolation_plancher_bas
8: adresse_ban
9: modele_dpe
10: zone_climatique
11: type_installation_chauffage
12: surface_habitable_logement
13: hauteur_sous_plafond
14: etiquette_dpe
15: _score
16: nombre_niveau_logement
17: isolation_toiture
18: type_generateur_n1_ecs_n1
19: type_generateur_chauffage_principal
20: type_generateur_froid
21: _outlier
22: TX
23: TN
24: periode_construction


In [53]:
# Afficher toutes les colonnes du DataFrame
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


0: conso_5_usages_ef
1: version_dpe
2: qualite_isolation_murs
3: type_batiment
4: classe_altitude
5: type_installation_ecs
6: type_energie_principale_chauffage
7: qualite_isolation_plancher_bas
8: adresse_ban
9: modele_dpe
10: zone_climatique
11: type_installation_chauffage
12: surface_habitable_logement
13: hauteur_sous_plafond
14: etiquette_dpe
15: _score
16: nombre_niveau_logement
17: isolation_toiture
18: type_generateur_n1_ecs_n1
19: type_generateur_chauffage_principal
20: type_generateur_froid
21: _outlier
22: TX
23: TN
24: periode_construction


In [54]:
# Variables explicatives et préparation des données
variables_explicatives = [
    'version_dpe',
    'qualite_isolation_murs',
    'type_batiment',
    'classe_altitude',
    'type_installation_ecs',
    'type_energie_principale_chauffage',
    'qualite_isolation_plancher_bas',
    'zone_climatique',
    'type_installation_chauffage',
    'surface_habitable_logement',
    'hauteur_sous_plafond',
    'nombre_niveau_logement',
    'isolation_toiture',
    'type_generateur_n1_ecs_n1',
    'type_generateur_chauffage_principal',
    'type_generateur_froid',
    'periode_construction'
]

variables_explicatives = [v for v in variables_explicatives if v in df.columns]

for col in variables_explicatives:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

X = pd.get_dummies(df[variables_explicatives], drop_first=True)
y = np.log1p(df[target])

print(f"Données prêtes : {X.shape}")
print(f"Statistiques log de la cible : mean={y.mean():.2f}, std={y.std():.2f}")


Données prêtes : (700000, 97)
Statistiques log de la cible : mean=9.10, std=0.87


In [55]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train : {X_train.shape[0]} lignes, Test : {X_test.shape[0]} lignes")


Train : 490000 lignes, Test : 210000 lignes


In [56]:
# Entraînement XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("Modèle XGBoost entraîné !")


Modèle XGBoost entraîné !


In [57]:
# Prédiction et métriques
y_pred = xgb_model.predict(X_test)

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred)

mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_real, y_pred_real)

print(f"Mean Squared Error (MSE) : {mse:.2f}")
print(f"Root MSE (RMSE) : {rmse:.2f}")
print(f"Coefficient de détermination (R²) : {r2:.3f}")


Mean Squared Error (MSE) : 1477154974.00
Root MSE (RMSE) : 38433.77
Coefficient de détermination (R²) : 0.490


In [58]:
#Sauvegarde du modèle
joblib.dump(xgb_model, 'modele_conso_xgb.pkl', compress=3)
print(" Modèle XGBoost enregistré dans 'modele_conso_xgb.pkl'.")


✅ Modèle XGBoost enregistré dans 'modele_conso_xgb.pkl'.
